# NB2 — Controlled local noise, baselines, misspecification, figures (E5, E6, E8, F)
Run **after NB1**: E6 reads NB1's `e3_counts.csv`. On Kaggle, attach NB1's output as a dataset
(or run both in one session) and set `NB1_OUT` below.

| Exp. | What | Output |
|---|---|---|
| E5 | Per-gate local noise (depolarizing; amplitude+phase damping) on all-to-all CZ circuits, n = 3–5: map p → λ_eff, model fit | `e5_local_noise.csv` |
| E6 | Baselines on the same noisy P̂(k): classical, standard Grover (k = R_opt), cost-optimal k chosen on held-out seeds, BBHT exponential search | `e6_baselines.csv` |
| E8 | Regret when k is chosen from a misspecified λ | `e8_misspec.csv` |
| F | Manuscript Figures 1–5 regenerated + new figures | `fig_*.png` |

In [ ]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "qiskit==2.6.1", "qiskit-aer==0.18.0", "qiskit-ibm-runtime==0.51.0"], check=False)

In [ ]:
import os, json, time, numpy as np, pandas as pd, matplotlib.pyplot as plt
QUICK = False
OUT = "/kaggle/working/results" if os.path.isdir("/kaggle/working") else "results"
import glob
_hits = glob.glob("/kaggle/input/**/e3_counts.csv", recursive=True)   # NB1 output attached via "Add Input"
NB1_OUT = os.path.dirname(_hits[0]) if _hits else OUT
print("NB1 results folder:", NB1_OUT)
os.makedirs(OUT, exist_ok=True)
SHOTS = 2048 if QUICK else 8192
SEEDS5 = [0, 1] if QUICK else [0, 1, 2]

In [ ]:
"""Shared code for the Grover-under-noise experiments (E1-E4).
Qiskit >= 2.0, qiskit-aer >= 0.17, qiskit-ibm-runtime >= 0.40 (fake backends)."""
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)  # QuantumCircuit.mcx(mode=...) deprecated in Qiskit 2.1; behaviour unchanged in 2.6.1
import numpy as np
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile

# ---------- theory (Eqs. 10, 13, 18-20 of the manuscript) ----------
def theta(N, M=1):
    return np.arcsin(np.sqrt(M / N))

def p_ideal(k, N, M=1):
    return np.sin((2 * np.asarray(k) + 1) * theta(N, M)) ** 2

def p_noisy(k, N, lam, M=1):
    q = 1.0 - lam
    k = np.asarray(k, dtype=float)
    return q ** k * p_ideal(k, N, M) + (1 - q ** k) * M / N

def r_opt(N, M=1):
    return int(round(np.pi / (4 * theta(N, M)) - 0.5))

# ---------- circuits ----------
def _mcz(qc, qubits, anc, mode):
    """Multi-controlled Z on `qubits` (last one is the target of the MCX)."""
    *ctrl, tgt = qubits
    qc.h(tgt)
    if len(ctrl) == 1:
        qc.cx(ctrl[0], tgt)
    elif len(ctrl) == 2:
        qc.ccx(ctrl[0], ctrl[1], tgt)
    elif mode == "v-chain" and anc is not None and len(anc) > 0:
        qc.mcx(ctrl, tgt, ancilla_qubits=list(anc), mode="v-chain")
    else:
        qc.mcx(ctrl, tgt)  # synthesised by the transpiler (ancilla-free)
    qc.h(tgt)

def grover_circuit(n, target, k, mode="noancilla", measure=True):
    """n-qubit single-target Grover circuit with k iterations.
    Qubit i carries bit 2**i, so the measured bitstring equals format(target, f'0{n}b')
    and x1 (MSB in the manuscript convention) is qubit n-1."""
    q = QuantumRegister(n, "q")
    regs = [q]
    anc = None
    if mode == "v-chain" and n >= 4:
        anc = QuantumRegister(n - 3, "a")
        regs.append(anc)
    if measure:
        regs.append(ClassicalRegister(n, "c"))
    qc = QuantumCircuit(*regs, name=f"grover_n{n}_k{k}")
    zeros = [i for i in range(n) if not (target >> i) & 1]
    qc.h(q)
    for _ in range(k):
        # oracle O = I - 2|x0><x0|
        if zeros:
            qc.x([q[i] for i in zeros])
        _mcz(qc, list(q), anc, mode)
        if zeros:
            qc.x([q[i] for i in zeros])
        # diffusion D = 2|psi0><psi0| - I  (up to global phase -1)
        qc.h(q); qc.x(q)
        _mcz(qc, list(q), anc, mode)
        qc.x(q); qc.h(q)
    if measure:
        qc.measure(q, regs[-1])
    return qc

# ---------- resource metrics ----------
TWO_Q = ("ecr", "cz", "cx")

def resources(tqc):
    ops = tqc.count_ops()
    n2q = sum(v for g, v in ops.items() if g in TWO_Q)
    active = {tqc.find_bit(qb).index for inst in tqc.data
              if inst.operation.name not in ("barrier", "delay") for qb in inst.qubits}
    d2 = tqc.depth(filter_function=lambda inst: inst.operation.num_qubits == 2)
    return dict(depth=tqc.depth(), depth_2q=d2, n_2q=n2q,
                n_1q=sum(v for g, v in ops.items() if g in ("rz", "sx", "x")),
                n_ops=sum(v for g, v in ops.items() if g not in ("barrier", "measure", "delay")),
                phys_qubits=len(active))

# ---------- statistics ----------
def wilson(s, n, z=1.959964):
    p = s / n
    den = 1 + z * z / n
    c = (p + z * z / (2 * n)) / den
    h = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / den
    return c - h, c + h

def _nll(params, ks, succ, shots, N, model):
    if model == "1p":
        lam, A = params[0], 1.0
    else:
        lam, A = params
    q = 1 - lam
    pk = A * q ** ks * p_ideal(ks, N) + (1 - A * q ** ks) / N
    pk = np.clip(pk, 1e-12, 1 - 1e-12)
    return -np.sum(succ * np.log(pk) + (shots - succ) * np.log(1 - pk))

def fit_lambda(ks, succ, shots, N, model="1p"):
    """Binomial MLE of lambda_eff (model '1p': Eq. 18) or (lambda, A) (model '2p':
    A <= 1 absorbs a k-independent SPAM/initialisation loss)."""
    from scipy.optimize import minimize, minimize_scalar
    ks, succ, shots = map(np.asarray, (ks, succ, shots))
    if model == "1p":
        r = minimize_scalar(lambda l: _nll([l], ks, succ, shots, N, "1p"),
                            bounds=(0.0, 0.999), method="bounded")
        return dict(lam=r.x, A=1.0, nll=r.fun)
    best = None
    for l0 in (0.05, 0.3, 0.7):
        r = minimize(lambda p: _nll(p, ks, succ, shots, N, "2p"), x0=[l0, 0.9],
                     bounds=[(0.0, 0.999), (0.0, 1.0)], method="L-BFGS-B")
        if best is None or r.fun < best.fun:
            best = r
    return dict(lam=best.x[0], A=best.x[1], nll=best.fun)

def expected_cost(ks, phat):
    phat = np.maximum(np.asarray(phat, float), 1e-12)
    return (np.asarray(ks) + 1) / phat

def speedup_from_cost(N, cost):
    return (N + 1) / 2 / np.min(cost)


## E5 — controlled local noise
Circuits transpiled (optimisation level 3, seed 0) to the basis {rz, sx, x, cz} with **all-to-all** connectivity,
so that only gate noise (no routing) enters. Noise channels per gate: (a) depolarizing, 2q error p, 1q error p/10;
(b) amplitude + phase damping after every 2q gate with γ = p on each qubit (1q gates noiseless).
For each (channel, n, p) λ_eff is fitted with models 1p and 2p as in NB1.

In [ ]:
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error, amplitude_damping_error, phase_damping_error
from scipy.stats import chi2
BASIS = ["rz", "sx", "x", "cz"]
def nm_depol(p):
    m = NoiseModel(basis_gates=BASIS)
    m.add_all_qubit_quantum_error(depolarizing_error(p, 2), ["cz"])
    m.add_all_qubit_quantum_error(depolarizing_error(p / 10, 1), ["sx", "x"])
    return m
def nm_damp(p):
    m = NoiseModel(basis_gates=BASIS)
    e1 = amplitude_damping_error(p).compose(phase_damping_error(p))
    m.add_all_qubit_quantum_error(e1.tensor(e1), ["cz"])
    return m
CHANNELS = {"local_depolarizing": nm_depol, "amp+phase_damping": nm_damp}
PS = [0.002, 0.005, 0.01, 0.02] if QUICK else [0.001, 0.002, 0.005, 0.01, 0.02, 0.05]
rows = []; t0 = time.time()
for n in [3, 4, 5]:
    N = 2 ** n; t = 107 % N; key = format(t, f"0{n}b"); K = 2 * r_opt(N) + 2
    circs = [transpile(grover_circuit(n, t, k, "v-chain"), basis_gates=BASIS,
                       optimization_level=3, seed_transpiler=0) for k in range(K + 1)]
    n2q_iter = (circs[K].count_ops().get("cz", 0) - circs[0].count_ops().get("cz", 0)) / K
    for ch, f in CHANNELS.items():
        for p in PS:
            sim = AerSimulator(noise_model=f(p))
            succ = np.zeros(K + 1); shots = np.zeros(K + 1)
            for s in SEEDS5:
                res = sim.run(circs, shots=SHOTS, seed_simulator=2000 + s).result()
                for k in range(K + 1):
                    succ[k] += res.get_counts(k).get(key, 0); shots[k] += SHOTS
            ks = np.arange(K + 1)
            f1, f2 = fit_lambda(ks, succ, shots, N, "1p"), fit_lambda(ks, succ, shots, N, "2p")
            ph = np.clip(succ / shots, 1e-12, 1 - 1e-12)
            sat = -np.sum(succ * np.log(ph) + (shots - succ) * np.log(1 - ph))
            dev = 2 * (f1["nll"] - sat)
            rows.append(dict(channel=ch, n=n, p=p, n2q_per_iter=n2q_iter, lam_1p=f1["lam"],
                             lam_2p=f2["lam"], A_2p=f2["A"], ratio_lam_over_n2q_p=f1["lam"] / (n2q_iter * p),
                             deviance_1p=dev, df=K, p_gof_1p=chi2.sf(dev, K),
                             p_LRT=chi2.sf(2 * (f1["nll"] - f2["nll"]), 1)))
        print(f"n={n} done ({time.time()-t0:.0f}s)")
e5 = pd.DataFrame(rows); e5.to_csv(f"{OUT}/e5_local_noise.csv", index=False); e5.round(4)

## E6 — baselines on identical noisy data (from NB1)
Selection uses **seeds 0–1 only** (fit λ̂, choose k*_C(λ̂)); all strategies are evaluated on **seeds 2–4**
so that the chosen k is not tuned on the evaluation data. BBHT: m ← 1, factor 6/5, cap √N; each trial picks
j uniformly in {0,…,⌈m⌉−1}, costs j + 1 queries (j iterations + verification) and succeeds with P̂_eval(j).
CIs: bootstrap over evaluation seeds with binomial resampling (300 resamples; BBHT 20 000 runs each).

In [ ]:
e3 = pd.read_csv(f"{NB1_OUT}/e3_counts.csv")
def bbht_cost(Ptab, N, rng, runs):
    kmax = len(Ptab) - 1; tot = 0
    for _ in range(runs):
        m, c = 1.0, 0
        while True:
            j = min(int(rng.integers(0, int(np.ceil(m)))), kmax); c += j + 1
            if rng.random() < Ptab[j]: break
            m = min(1.2 * m, np.sqrt(N))
        tot += c
    return tot / runs
sel_seeds = [0, 1]; ev_seeds = sorted(set(e3.seed.unique()) - set(sel_seeds))
assert ev_seeds, "need >= 3 seeds in NB1 (set QUICK=False)"
rows = []
for (b, n), g in e3.groupby(["backend", "n"]):
    N = 2 ** n; R = r_opt(N)
    gs = g[g.seed.isin(sel_seeds)].groupby("k")[["succ", "shots"]].sum()
    lam_sel = fit_lambda(gs.index.values, gs.succ.values, gs.shots.values, N, "1p")["lam"]
    kgrid = np.arange(0, len(gs)); k_ours = int(kgrid[np.argmin((kgrid + 1) / p_noisy(kgrid, N, lam_sel))])
    rng = np.random.default_rng(7 + n); boots = {s: [] for s in ["classical", "grover_Ropt", "ours_heldout", "best_k_insample", "BBHT"]}
    ge = g[g.seed.isin(ev_seeds)]
    for bi in range(1 if QUICK else 300):
        pick = rng.choice(ev_seeds, len(ev_seeds), replace=True) if bi else ev_seeds
        gb = pd.concat([ge[ge.seed == s] for s in pick]).groupby("k")[["succ", "shots"]].sum()
        P = (rng.binomial(gb.shots.values, gb.succ.values / gb.shots.values) / gb.shots.values) if bi else gb.succ.values / gb.shots.values
        cost = expected_cost(gb.index.values, P)
        boots["classical"].append((N + 1) / 2)
        boots["grover_Ropt"].append(cost[R]); boots["ours_heldout"].append(cost[k_ours])
        boots["best_k_insample"].append(cost.min())
        boots["BBHT"].append(bbht_cost(P, N, rng, 2000 if QUICK else 20000) if bi % 10 == 0 else np.nan)
    for s, v in boots.items():
        v = np.array(v, float); v_ok = v[~np.isnan(v)]
        rows.append(dict(backend=b, n=n, N=N, strategy=s, k_used={"grover_Ropt": R, "ours_heldout": k_ours}.get(s, np.nan),
                         lam_sel=lam_sel, exp_queries=v[0], lo=np.percentile(v_ok, 2.5), hi=np.percentile(v_ok, 97.5),
                         speedup=(N + 1) / 2 / v[0]))
e6 = pd.DataFrame(rows); e6.to_csv(f"{OUT}/e6_baselines.csv", index=False)
e6.pivot_table(index=["backend", "n"], columns="strategy", values="exp_queries").round(2)

## E4b — predictions from the two-parameter model
NB1 shows that the one-parameter model (Eq. 18) is rejected and over-predicts the speedup. Here the
two-parameter fit (λ, A) from NB1 (`e3_fits.csv`) is used to predict k*_C and S, and both predictions are
compared with the empirical values and their bootstrap CIs (`e4_summary.csv`).

In [ ]:
fits = pd.read_csv(f"{NB1_OUT}/e3_fits.csv"); e4s = pd.read_csv(f"{NB1_OUT}/e4_summary.csv")
rows = []
for _, r in fits.iterrows():
    N = int(r.N); kg = np.arange(0, 4 * int(np.sqrt(N)) + 21); q = 1 - r.lam_2p
    P2 = r.A_2p * q ** kg * p_ideal(kg, N) + (1 - r.A_2p * q ** kg) / N
    c2 = (kg + 1) / P2
    s = e4s[(e4s.backend == r.backend) & (e4s.n == r.n)].iloc[0]
    rows.append(dict(backend=r.backend, n=int(r.n), kC_emp=int(s.kC_emp), kC_pred_1p=int(s.kC_pred_fit),
                     kC_pred_2p=int(kg[np.argmin(c2)]), kC_pred_cal=int(s.kC_pred_cal),
                     S_hat=s.S_hat, S_lo=s.S_lo, S_hi=s.S_hi, S_pred_1p=s.S_pred_fit,
                     S_pred_2p=(N + 1) / 2 / c2.min(), S_pred_cal=s.S_pred_cal,
                     in_CI_1p=bool(s.S_lo <= s.S_pred_fit <= s.S_hi),
                     in_CI_2p=bool(s.S_lo <= (N + 1) / 2 / c2.min() <= s.S_hi)))
e4b = pd.DataFrame(rows); e4b.to_csv(f"{OUT}/e4b_model_predictions.csv", index=False)
print("k*_C correct  1p:", (e4b.kC_pred_1p == e4b.kC_emp).sum(), " 2p:", (e4b.kC_pred_2p == e4b.kC_emp).sum(),
      " cal:", (e4b.kC_pred_cal == e4b.kC_emp).sum(), " of", len(e4b))
print("S inside empirical 95% CI  1p:", e4b.in_CI_1p.sum(), " 2p:", e4b.in_CI_2p.sum())
e4b.round(3)

## E4c — held-out model check
Models fitted on seeds 0–1 only; empirical optimum and speedup taken from seeds 2–4. Also reports the cost regret of the
1p-chosen k and the expected number of native two-qubit gates to success (query cost ≠ gate cost).

In [ ]:
rows = []
for (b, n), g in e3.groupby(["backend", "n"]):
    N = 2 ** n
    tr = g[g.seed.isin([0, 1])].groupby("k")[["succ", "shots"]].sum()
    te = g[g.seed.isin([2, 3, 4])].groupby("k").agg(succ=("succ", "sum"), shots=("shots", "sum"), n2q=("n_2q", "mean"))
    f1 = fit_lambda(tr.index.values, tr.succ.values, tr.shots.values, N, "1p")
    f2 = fit_lambda(tr.index.values, tr.succ.values, tr.shots.values, N, "2p")
    ks = te.index.values; ph = (te.succ / te.shots).values; c = (ks + 1) / np.maximum(ph, 1e-12)
    kg = np.arange(0, 4 * int(np.sqrt(N)) + 21)
    c1 = (kg + 1) / p_noisy(kg, N, f1["lam"]); q = 1 - f2["lam"]
    c2 = (kg + 1) / (f2["A"] * q ** kg * p_ideal(kg, N) + (1 - f2["A"] * q ** kg) / N)
    k1, k2 = int(kg[np.argmin(c1)]), int(kg[np.argmin(c2)]); S_te = (N + 1) / 2 / c.min()
    rows.append(dict(backend=b, n=n, k_emp_test=int(ks[np.argmin(c)]), k_1p=k1, k_2p=k2, S_test=S_te,
                     S_1p=(N + 1) / 2 / c1.min(), S_2p=(N + 1) / 2 / c2.min(),
                     regret_1p=c[k1] / c.min() - 1, exp_queries_1p=c[k1], exp_2q_gates_1p=te.n2q.values[k1] / ph[k1]))
e4c = pd.DataFrame(rows)
e4c["relerr_S_1p"] = (e4c.S_1p - e4c.S_test) / e4c.S_test; e4c["relerr_S_2p"] = (e4c.S_2p - e4c.S_test) / e4c.S_test
e4c.to_csv(f"{OUT}/e4c_heldout.csv", index=False)
print("held-out k correct 1p:", (e4c.k_1p == e4c.k_emp_test).sum(), " 2p:", (e4c.k_2p == e4c.k_emp_test).sum(), " of", len(e4c))
print("mean rel. error of S  1p: %+.3f  2p: %+.3f ; max regret 1p: %.3f" % (e4c.relerr_S_1p.mean(), e4c.relerr_S_2p.mean(), e4c.regret_1p.max()))
e4c.round(3)

## E8 — misspecified λ (theory, Eq. 18–20)
k is chosen as the cost minimiser for λ̂ = (1+δ)λ; regret = C_λ(k(λ̂)) / min_k C_λ(k) − 1.

In [ ]:
rows = []
for N in [16, 128, 1024, 2 ** 14]:
    kg = np.arange(0, 4 * int(np.sqrt(N)) + 21)
    for lam in [0.005, 0.01, 0.03, 0.1, 0.3]:
        c_true = (kg + 1) / p_noisy(kg, N, lam)
        for d in [-0.75, -0.5, -0.25, 0.25, 0.5, 1.0, 3.0]:
            lh = min(lam * (1 + d), 0.999)
            kh = kg[np.argmin((kg + 1) / p_noisy(kg, N, lh))]
            rows.append(dict(N=N, lam=lam, delta=d, k_true=int(kg[np.argmin(c_true)]), k_chosen=int(kh),
                             regret=c_true[kh] / c_true.min() - 1))
e8 = pd.DataFrame(rows); e8.to_csv(f"{OUT}/e8_misspec.csv", index=False)
e8[e8.N == 128].pivot_table(index="lam", columns="delta", values="regret").round(3)

## F — figures (manuscript Figs. 1–5 regenerated from code, plus new figures)

In [ ]:
N = 128; ks = np.arange(11); th = theta(N)
a = np.sin((2 * ks + 1) * th); b = np.cos((2 * ks + 1) * th) / np.sqrt(N - 1)
fig, ax = plt.subplots(figsize=(5, 3.2)); ax.plot(ks, a, "o-", label="$a_k$ (marked)"); ax.plot(ks, b, "s-", label="$b_k$ (unmarked)")
ax.axvline(8, ls="--", c="grey"); ax.set_xlabel("k"); ax.set_ylabel("amplitude"); ax.legend(); fig.tight_layout(); fig.savefig(f"{OUT}/fig1_amplitudes.png", dpi=300)
fig, ax = plt.subplots(figsize=(5, 3.2)); ax.plot(ks, a ** 2, "o-", label="P(k)"); ax.plot(ks, 1 - a ** 2, "s-", label="$P_{rest}(k)$")
ax.set_xlabel("k"); ax.set_ylabel("probability"); ax.legend(); fig.tight_layout(); fig.savefig(f"{OUT}/fig2_probabilities.png", dpi=300)
fig, ax = plt.subplots(figsize=(5, 3.2)); ax.plot(ks, p_ideal(ks, N), "o-", label="ideal"); ax.plot(ks, p_noisy(ks, N, 0.03), "s-", label="λ = 0.03")
ax.set_xlabel("k"); ax.set_ylabel("success probability"); ax.legend(); fig.tight_layout(); fig.savefig(f"{OUT}/fig3_noisy_probability.png", dpi=300)
Nc = 2 ** np.linspace(3, 7, 200); fig, ax = plt.subplots(figsize=(5, 3.2))
ax.plot(Nc, np.pi / (4 * np.arcsin(Nc ** -0.5)) - 0.5, label="continuous"); Ni = 2 ** np.arange(3, 8)
ax.plot(Ni, [r_opt(x) for x in Ni], "o", label="$R_{opt}$ (Eq. 13)"); ax.plot([128], [8], "r*", ms=12)
ax.set_xscale("log", base=2); ax.set_xlabel("N"); ax.set_ylabel("iterations"); ax.legend(); fig.tight_layout(); fig.savefig(f"{OUT}/fig4_Ropt.png", dpi=300)
def S_of(N, lam):
    kg = np.arange(0, int(4 * np.sqrt(N)) + 21); return (N + 1) / 2 / ((kg + 1) / p_noisy(kg, N, lam)).min()
Ns = 2 ** np.arange(3, 21); fig, ax = plt.subplots(figsize=(5, 3.4))
ax.plot(Ns, [S_of(x, 0) for x in Ns], "o-", label="λ = 0"); ax.plot(Ns, [S_of(x, 1 / (np.pi * np.sqrt(x))) for x in Ns], "s-", label="λ = 1/(π√N)")
ax.plot(Ns, [S_of(x, 0.03) for x in Ns], "^-", label="λ = 0.03"); ax.axhline(24.17, ls=":", c="grey"); ax.axvline(128, ls="--", c="grey")
ax.set_xscale("log", base=2); ax.set_yscale("log"); ax.set_xlabel("N"); ax.set_ylabel("speedup S(λ, N)"); ax.legend(fontsize=8); fig.tight_layout(); fig.savefig(f"{OUT}/fig5_speedup_vs_N.png", dpi=300)
# new: E6 expected queries
fig, axes = plt.subplots(1, e6.backend.nunique(), figsize=(12, 3.4), sharey=True)
for ax, (bk, d) in zip(np.atleast_1d(axes), e6.groupby("backend")):
    for s, ds in d.groupby("strategy"):
        ax.errorbar(ds.n, ds.exp_queries, yerr=[ds.exp_queries - ds.lo, ds.hi - ds.exp_queries], fmt="o-", ms=3, label=s, capsize=2)
    ax.set_yscale("log"); ax.set_title(bk, fontsize=9); ax.set_xlabel("n")
np.atleast_1d(axes)[0].set_ylabel("expected oracle queries"); np.atleast_1d(axes)[0].legend(fontsize=7); fig.tight_layout(); fig.savefig(f"{OUT}/fig_e6_baselines.png", dpi=300)
# new: E5 lambda_eff vs p
fig, ax = plt.subplots(figsize=(5, 3.4))
for (ch, n), d in e5.groupby(["channel", "n"]):
    ax.plot(d.p, d.lam_1p, "o-" if ch.startswith("local") else "s--", label=f"{ch}, n={n}")
ax.set_xscale("log"); ax.set_xlabel("per-gate error p"); ax.set_ylabel("fitted λ_eff"); ax.axhline(0.625, c="r", ls=":", lw=1)
ax.legend(fontsize=6); fig.tight_layout(); fig.savefig(f"{OUT}/fig_e5_lambda_vs_p.png", dpi=300)
print(sorted(f for f in os.listdir(OUT) if f.endswith('.png')))